In [ ]:
import psycopg2
import duckdb

In [ ]:
con = duckdb.connect()
for ext in ["postgres", "spatial"]:
    con.install_extension(ext)
    con.load_extension(ext)
con.sql("""
    ATTACH 'dbname=spartid_ais user=postgres password=postgres host=127.0.0.1 port=5433' AS db (TYPE postgres);
""")

In [ ]:
# Connection parameters
DB_CONFIG = {
    'host': 'localhost',  # or your TimescaleDB host
    'port': 5433,         # default PostgreSQL port
    'database': 'spartid_ais',
    'user': 'postgres',
    'password': 'postgres'
}

In [ ]:
def connect_psycopg2():
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        print("Connected to TimescaleDB successfully!")
        return conn
    except psycopg2.Error as e:
        print(f"Error connecting to TimescaleDB: {e}")
        return None

In [ ]:
conn = connect_psycopg2()
if conn:
    cursor = conn.cursor()

    # Create a hypertable (TimescaleDB specific)
    create_hypertable_sql = """
    CREATE TABLE historic_position (
        id integer NOT NULL,
        msg_type smallint NOT NULL,
        repeat smallint NOT NULL,
        mmsi integer NOT NULL,
        status varchar NOT NULL,
        turn double precision NOT NULL,
        speed double precision NOT NULL,
        accuracy boolean NOT NULL,
        lat double precision NOT NULL,
        long double precision NOT NULL,
        course double precision NOT NULL,
        heading integer NOT NULL,
        maneuver varchar NOT NULL,
        raim boolean NOT NULL,
        radio integer NOT NULL,
        "timestamp" timestamp without time zone NOT NULL
    );

    -- Convert to hypertable (TimescaleDB extension)
    SELECT create_hypertable('historic_position', by_range('timestamp', INTERVAL '1 day'), if_not_exists => TRUE);
    """

    try:
        cursor.execute(create_hypertable_sql)
        conn.commit()
        print("Hypertable created successfully!")
    except psycopg2.Error as e:
        print(f"Error creating hypertable: {e}")

    cursor.close()
    conn.close()

In [ ]:
# conn = connect_psycopg2()
# cursor = conn.cursor()
# cursor.execute("DROP TABLE last_position;")
# conn.commit()
# cursor.close()
# conn.close()


In [ ]:
# con.sql("SELECT * FROM db.public.last_position ORDER BY timestamp DESC").df()
con.sql("SELECT * FROM db.public.historic_position ORDER BY timestamp DESC LIMIT 10").df()

In [ ]:
con.sql("""
        INSERT INTO db.public.historic_position
            SELECT
                id, msg_type, repeat, mmsi, status, turn, speed, accuracy, lat, long, course, heading, maneuver, raim, radio, "timestamp"
            FROM read_parquet('historic_position_2024_w16.parquet')
""")

In [ ]:
con.sql("DESCRIBE SELECT * FROM read_parquet('historic_position_2024_w10.parquet')")

In [ ]:
con.sql("""
        SELECT COUNT(*)
        FROM db.historic_position
        WHERE mmsi = '257330900' AND
            "timestamp" >= '2024-03-11'::timestamp AND
            "timestamp" < '2024-03-31'::timestamptz;
""")

In [ ]:
conn = connect_psycopg2()
cursor = conn.cursor()
res = cursor.execute("""
        SELECT COUNT(*)
        FROM historic_position
        WHERE mmsi = '257330900' AND
            "timestamp" >= '2024-03-11'::timestamp AND
            "timestamp" < '2024-03-31'::timestamptz;
""")
print(cursor.fetchall())
cursor.close()
conn.close()
